# 02 · Cosmology

Along a photon path the scale factor obeys da/dx = (H0/c) √(Ω_r + Ω_m a + Ω_k a² + Ω_Λ a⁴) / √(1 − k x²)
(review notes, Eq. dadx_general). `Cosmology` holds a batch of backgrounds and integrates it with
`diffrax`. The skewers (03) use it to solve a(x) along every path. Runtime: ~10 s.

**Requires:** nothing.  
**Independent of:** lines, skewers, fields.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('cosmology',)); plt.show()

In [ ]:
from skewerjax.core import Cosmology, phys
from skewerjax.utils.plotting import plot_scale_factor_solutions
h = phys.H0_over_c                                                  # H0 / c [1/Mpc]
names = ["radiation", "matter", "Lambda", "Planck 2018"]
cosmo = Cosmology(H0=jnp.full(4, phys.H0), Omega_r_0=jnp.array([1.0, 0.0, 0.0, 9.1436e-5]), Omega_m_0=jnp.array([0.0, 1.0, 0.0, 0.3089]),
                  Omega_k_0=jnp.zeros(4), Omega_lambda_0=jnp.array([0.0, 0.0, 1.0, 0.6911]))
x0, a0 = 0.0, 1.0                                                   # initial conditions: the observer today
x_f, x_b = np.linspace(x0, x0 + 1 / (a0 * h) - 1e-6, 600), np.linspace(x0, x0 - 2 * np.sqrt(a0) / h + 1e-6, 600)
a = np.concatenate([np.asarray(cosmo.solve_a_of_x(jnp.full(4, a0), jnp.asarray(x_b)))[:, ::-1],
                    np.asarray(cosmo.solve_a_of_x(jnp.full(4, a0), jnp.asarray(x_f)))], 1)
x = np.concatenate([x_b[::-1], x_f])
exact = {"radiation": a0 + h * (x - x0),                             # da/dx = h
         "matter": (np.sqrt(a0) + 0.5 * h * (x - x0)) ** 2,          # da/dx = h a^(1/2)
         "Lambda": a0 / (1 - a0 * h * (x - x0)), "Planck 2018": None} # da/dx = h a^2
colors = dict(zip(names, [TOL["red"], TOL["blue"], TOL["green"], FG]))
fig, axs = plot_scale_factor_solutions(x, dict(zip(names, a)), exact, colors, x0=x0, a0=a0, h=h)
add_panel_letters(axs)
fig.suptitle("a(x) along a light ray: the ODE solution on the closed forms (thick, light), and where each one ends", fontsize=22)
plt.show()
for name, xs in (("radiation", x0 - a0 / h), ("matter", x0 - 2 * np.sqrt(a0) / h), ("Lambda", x0 + 1 / (a0 * h))):
    i = names.index(name)
    near = np.argmin(np.abs(x - xs))
    print(f"{name:9s}: special point x = {xs:9.1f} Mpc; numerical a there = {a[i, near]:.3e}; "
          f"max |a / exact - 1| = {np.nanmax(np.abs(a[i][exact[name] > 0] / exact[name][exact[name] > 0] - 1)):.1e}")

*Reading the figure.* With a single component, da/dx = (H0/c) √(Ω a^p) has closed forms (initial conditions a0
at x0, the observer today, cross and dotted guide lines):
- **radiation** (p = 0): a = a0 + (H0/c)(x − x0) vanishes at **x = x0 − a0/(H0/c)**, the big bang along this ray;
- **matter** (p = 1): a = [a0^{1/2} + (H0/c)(x − x0)/2]² reaches a = a′ = 0 at **x = x0 − 2a0^{1/2}/(H0/c)**;
- **Λ** (p = 4): a = a0/[1 − a0(H0/c)(x − x0)] diverges at **x = x0 + 1/(a0 H0/c)**, a finite comoving distance to
  the future.

The vertical lines mark these points in each background's colour; Planck 2018 lies between them. The residual (b) is
on a symmetric log scale (linear below 10⁻⁵), so the ODE error of every background is visible: it stays near the
solver tolerance and grows only next to the special points, where a → 0 or a → ∞.

The default tolerances (`rtol = atol = 1e-4`) give a(x) to ~10⁻⁴; `solve_a_of_x(..., rtol=1e-10, atol=1e-12)` to ~10⁻⁹ (`scripts/validation/check_cosmology.py`). The parameters are pytree leaves: `jax.grad` of any result with respect to Ω_m works.